In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nltk
import time
import os
import re
import warnings
from tqdm import tqdm
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
from transformers import BertModel, BertTokenizer, AdamW, get_linear_schedule_with_warmup
import nltk
import requests
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve, roc_auc_score
import tempfile
import torchmetrics

In [2]:
warnings.filterwarnings('ignore')
device = torch.device("cuda")

## Loading data

In [ ]:
def get_category_index(category):

    categories = {
    'нет конкретного ответа' : 0,
    '?' : 0,
    'ок' : 1,
    'work-life balance' : 2,
    'адекватные планы и количество метрик' : 3,
    'адекватные планы и кол-во метрик' : 3,
    'бесплатное питание' : 4,
    'бесплатная еда' : 4,
    'бюрократия' : 5,
    'взаимодействие' : 6,
    'взаимодействие ' : 6,
    'внерабочие активности' : 7,
    'график работы' : 8,
    'график' : 8,
    'дополнительные сотрудники' : 9,
    'идея по продукту' : 10,
    'идеи по продукту' : 10,
    'карьерный рост' : 11,
    'клиенты' : 12,
    'конкурсы' : 13,
    'культура обратной связи' : 14,
    'культура обратной связи ' : 14,
    'лояльность к сотрудникам' : 15,
    'льготы' : 16,
    'ль' : 16,
    'спортивный зал' : 16,
    'бассейн' : 16,
    'мерч' : 17,
    'нездоровая атмосфера' : 18,
    'обучение' : 19,
    'оплата труда' : 20,
    'оплата' : 20,
    'оплата трудв' : 20,
    'офисное пространство' : 21,
    'подарки на праздники' : 22,
    'подарки по праздникам' : 22,
    'подарки детям' : 22,
    'премии' : 23,
    'Премии' : 23,
    'процессы' : 24,
    'сложность работы' : 25,
    'техника/ит' : 26,
    'технологии/ит' : 26,
    'удаленная работа' : 27,
    'работа из заграницы' : 27,
    'работа из других стран' : 27,
    'оплата сверхурочного труда' : 28,
    'руководитель' : 29
    }

    try:
        return categories[category]
    except KeyError:
        return None

def get_category_name(index):

    categories_from_indices = {
    0 :  'нет конкретного ответа',
    1 :  'ок',
    2 :  'work-life balance',
    3 :  'адекватные планы и количество метрик',
    4 :  'бесплатное питание',
    5 :  'бюрократия',
    6 :  'взаимодействие',
    7 :  'внерабочие активности',
    8 :  'график работы',
    9 :  'дополнительные сотрудники',
    10 :  'идея по продукту',
    11 :  'карьерный рост',
    12 :  'клиенты',
    13 :  'конкурсы',
    14 :  'культура обратной связи',
    15 :  'лояльность к сотрудникам',
    16 :  'льготы',
    17 :  'мерч',
    18 :  'нездоровая атмосфера',
    19 :  'обучение',
    20 :  'оплата труда',
    21 :  'офисное пространство',
    22 :  'подарки на праздники',
    23 :  'премии',
    24 :  'процессы',
    25 :  'сложность работы',
    26 :  'техника/ит',
    27 : 'удаленная работа',
    28 : 'оплата сверхурочного труда',
    29 : 'руководитель'
    }

    try:
        return categories_from_indices[index]
    except KeyError:
        return None

def load_data(filename):
    if filename.endswith('xlsx'):
        df = pd.read_excel(filename)
    elif filename.endswith('csv'):
        df = pd.read_csv(filename)
    df1 = df.loc[:,['Score','A1','C1']].rename(columns={'Score':'S', 'A1':'A', 'C1':'C'})
    df2 = df.loc[:,['Score','A2','C2']].rename(columns={'Score':'S', 'A2':'A', 'C2':'C'})
    df = pd.concat([df1,df2]).dropna().reset_index(drop=True)
    df['Y'] = df['C'].apply(get_category_index)
    df['C'] = df['Y'].apply(get_category_name)
    return df

def delete_non_alpha(text):
    global stopwords
    text = str(text)
    text = text.lower()
    text = re.sub(r'[^a-zа-яё]', ' ', str(text))
    text = re.sub(r'\b\w\b', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def get_full_dict():
    url = "https://raw.githubusercontent.com/danakt/russian-words/master/russian.txt"
    response = requests.get(url)
    response.raise_for_status()
    return set(response.text.splitlines())

def setup_symspell():

    url = "https://raw.githubusercontent.com/wolfgarbe/SymSpell/master/SymSpell.FrequencyDictionary/ru-100k.txt"
    response = requests.get(url)
    response.raise_for_status()

    with tempfile.NamedTemporaryFile(mode="w+", delete=False, encoding="utf-8") as f:
        for line in response.text.splitlines():
            word, freq = line.split()
            f.write(f"{word}\t{freq}\n")
        temp_path = f.name

    sym_spell = SymSpell()
    sym_spell.load_dictionary(
        temp_path, 
        term_index=0, 
        count_index=1,
        separator="\t")
    
    return sym_spell

def get_stopwords():
    try:
        stopwords = nltk.corpus.stopwords.words("russian")
    except ModuleNotFoundError:
        nltk.download('stopwords')
        stopwords = nltk.corpus.stopwords.words("russian")
    antistopwords = ['не','нет','ни','ничего','без','никогда','нельзя','всегда',
                     'конечно','надо','хорошо','лучше','больше','более']
    for word in antistopwords:
        stopwords.remove(word)
    return set(stopwords)

def correct_text(words, full_dict, sym_spell):
    corrected = []
    for word in words:
        if not word in full_dict:
            suggestions = sym_spell.lookup(word, Verbosity.TOP)
            if suggestions:
                corrected.append(suggestions[0].term)
            else:
                corrected.append(word)
        else:
            corrected.append(word)
    return corrected

def lemmatize(words, morph):
    lemmatized = []
    for word in words:
        lemmatized.append(morph.parse(word)[0].normal_form)
    return lemmatized

def delete_stop_words(words, stopwords):
    normal_words = []
    for word in words:
        if not word in stopwords:
            normal_words.append(word)
    return normal_words

def check_empty(words):
    if len(words) == 0:
        return None
    return words

def text_from_words(words):
    if words:
        return ' '.join(word for word in words)
    return None

def prepare_data(df, test_size=0.25,  syntax_correction=False, lemmatization=False, stopwords_removal=False):
    
    A = np.array(df['A'])
    A = [delete_non_alpha(text) for text in tqdm(A, desc='Deleting non-alpha characters')]
    A = [text.split() for text in A]

    num_words = [len(words) for words in A]

    if syntax_correction:
        sym_spell = setup_symspell()
        full_dict = get_full_dict()
        A = [correct_text(words,full_dict,sym_spell) for words in tqdm(A, desc='Correcting syntax')]
    
    if stopwords_removal:
        stopwords = get_stopwords()
        A = [delete_stop_words(words, stopwords) for words in tqdm(A, desc='Deleting stopwords')]
    
    A = [check_empty(words) for words in tqdm(A, desc='Checking empty text')]
    A = [text_from_words(words) for words in A]
    df['A'] = A
    df = df.dropna()
    df_train, df_test = train_test_split(df, test_size=test_size, random_state=42, stratify=df['Y'])
   
    return df_train, df_test

def prepare_inference_data(df, col, syntax_correction=False, lemmatization=False, stopwords_removal=False):
    df_i = df.loc[:, ['Score', col]].copy()
        
    A = np.array(df_i[col])
    A = [delete_non_alpha(text) for text in tqdm(A, desc='Deleting non-alpha characters')]
    A = [text.split() for text in A]

    num_words = [len(words) for words in A]

    if syntax_correction:
        sym_spell = setup_symspell()
        full_dict = get_full_dict()
        A = [correct_text(words,full_dict,sym_spell) for words in tqdm(A, desc='Correcting syntax')]
    
    if stopwords_removal:
        stopwords = get_stopwords()
        A = [delete_stop_words(words, stopwords) for words in tqdm(A, desc='Deleting stopwords')]
    
    A = [check_empty(words) for words in tqdm(A, desc='Checking empty text')]
    A = [text_from_words(words) for words in A]
    df_i[col] = A
    df_i[col] = df_i[col].replace('nan', np.nan)
    df_i = df_i.dropna()
        
    return df_i
    

In [ ]:
df = pd.read_csv('../data/enps_nov_2025.csv') # new test dataset
df1 = prepare_inference_data(df, 'A1')
df2 = prepare_inference_data(df, 'A2')

Checking empty text: 100%|██████████| 5512/5512 [00:00<00:00, 2563935.19it/s]


## Tokenization

In [5]:
tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-uncased', do_lower_case=True)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/872k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.72M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

In [22]:
MAX_SUBTOKENS = 4

bad_words = set()

for text in df1['A1'].dropna().astype(str):
    words = re.findall(r"\S+", text)

    for w in words:
        n_sub = len(tokenizer.tokenize(w))
        if n_sub > MAX_SUBTOKENS:
            bad_words.add(w)

for text in df2['A2'].dropna().astype(str):
    words = re.findall(r"\S+", text)

    for w in words:
        n_sub = len(tokenizer.tokenize(w))
        if n_sub > MAX_SUBTOKENS:
            bad_words.add(w)

print(f"Найдено слов: {len(bad_words)}")
print(list(bad_words)[:20])


Найдено слов: 2550
['дрработодателями', 'амбизиозный', 'выполняешь', 'администратор', 'автоматизированная', 'микроволновка', 'промежуточные', 'заблаговременно', 'решаютмся', 'ежеквартально', 'комменариев', 'неэффективно', 'вознагрождение', 'некорректное', 'беспроводные', 'руководтсва', 'конфлюенс', 'увольняют', 'бесполезный', 'объязательно']


In [23]:
output_path = "bad_words_4.txt"

with open(output_path, "w", encoding="utf-8") as f:
    for word in sorted(bad_words):
        f.write(word + "\n")

In [6]:
def preprocessing_for_bert(data, max_len=64):
    
    input_ids = []
    attention_masks = []

    for text in tqdm(data):

        encoded_sent = tokenizer.encode_plus(
            text=text,  
            add_special_tokens=True,       
            max_length=max_len,                  
            pad_to_max_length=True,       
            truncation=True,         
            return_attention_mask=True 
            )
        
        input_ids.append(encoded_sent.get('input_ids'))
        attention_masks.append(encoded_sent.get('attention_mask'))

    input_ids = torch.tensor(input_ids)
    attention_masks = torch.tensor(attention_masks)

    return input_ids, attention_masks

In [7]:
inputs1, masks1 = preprocessing_for_bert(np.array(df1['A1']))
inputs2, masks2 = preprocessing_for_bert(np.array(df2['A2']))

100%|██████████| 3185/3185 [00:01<00:00, 1866.93it/s]


In [9]:
# inputs1, masks1 = preprocessing_for_bert(np.array(df1['A1']))
# inputs2, masks2 = preprocessing_for_bert(np.array(df2['A2']))

scores1 = torch.tensor(np.array(df1['Score']), dtype=torch.float32)
scores2 = torch.tensor(np.array(df2['Score']), dtype=torch.float32)

batch_size = 64

data1 = TensorDataset(inputs1, masks1, scores1)
sampler1 = SequentialSampler(data1)
dataloader1 = DataLoader(data1, sampler=sampler1, batch_size=batch_size)

data2 = TensorDataset(inputs2, masks2, scores2)
sampler2 = SequentialSampler(data2)
dataloader2 = DataLoader(data2, sampler=sampler2, batch_size=batch_size)

## Model

In [10]:
bert = BertModel.from_pretrained('bert-base-multilingual-uncased')

model.safetensors:   0%|          | 0.00/672M [00:00<?, ?B/s]

In [ ]:
class BERTClassifier(nn.Module):
    
    def __init__(self, bert, task='binary', n_classes=None, device='cuda'):

        super(BERTClassifier, self).__init__()
        
        self.bert = bert
        self.task = task
        self.n_classes = n_classes
        self.dropout = nn.Dropout(0.1)
        self.silu =  nn.SiLU()
        self.fc1 = nn.Linear(768, 128)
        self.fcs1 = nn.Linear(1, 32)
        self.fcs2 = nn.Linear(32, 64)
        self.fc2 = nn.Linear(192, 1) if self.task=='binary' else nn.Linear(192, n_classes)
        self.output = nn.Sigmoid() if self.task=='binary' else nn.Softmax()
        self.best_metric = 0
        self.epoch = 0
        self.history = {'loss' : {'train_loss' : [], 'val_loss' : []}, 
                        'metrics' : {'precision': [], 'recall': [], 'specificity' : [], 'f1' : []}}
        self.device = device
        self.to(device)

    def forward(self, x, s, mask):
  
        _, x = self.bert(x, attention_mask=mask).values()
        x = self.fc1(x)
        x = self.silu(x)
        s = self.fcs1(s.unsqueeze(1))
        s = self.silu(s)
        s = self.fcs2(s)
        s = self.silu(s)
        x = torch.concat((x,s), axis=1)
        x = self.dropout(x)
        x = self.fc2(x)
        x = self.output(x)
        
        return x

    def predict(self, data, thresh=0.5):
        
        self.eval()
        predictions = []
        
        with tqdm(total = len(data)) as tq:
            for batch in data:
                b_data = tuple(t.to(self.device) for t in batch)
                if len(b_data) == 4:
                    b_input_ids, b_attn_mask, b_scores, b_labels = b_data
                else:
                    b_input_ids, b_attn_mask, b_scores = b_data
                with torch.no_grad():
                    preds = self.forward(b_input_ids, b_scores, b_attn_mask).cpu()
                predictions.extend(preds)
                tq.update(1)
                
        predictions = np.array(predictions)

        if self.task == 'binary':
            return (predictions.reshape(-1) > thresh).astype(int)
            
        return predictions.argmax(axis=1).astype(int)

    def fit(self, train_dataloader, val_dataloader=None, val_metric = 'f1', epochs=5, early_stopping = None, evaluation=True, save_best=True):

        epochs_without_improvements = 0
        
        for epoch in range(self.epoch, self.epoch + epochs):
    
            t_start = time.time()
    
            total_loss = 0.
    
            self.train()
    
            with tqdm(total = len(train_dataloader)) as tq:
                
                for step, batch in enumerate(train_dataloader):
                    
                    self.optimizer.zero_grad()
                    b_input_ids, b_attn_mask, b_scores, b_labels = tuple(t.to(self.device) for t in batch)
                    b_labels = b_labels.reshape(-1,1)
                    pred = self.forward(b_input_ids, b_scores, b_attn_mask)
                    loss = self.loss_fn(pred, b_labels)
                    loss_value = loss.item()
                    total_loss += loss_value
        
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(self.parameters(), 1.0)
                    self.optimizer.step()
                    self.scheduler.step()
                    tq.update(1)
                    tq.set_description("Loss %s" % loss_value)
            
    
            avg_train_loss = total_loss / len(train_dataloader)
            
            self.history['loss']['train_loss'].append(avg_train_loss)

            self.epoch += 1

            epochs_without_improvements += 1
    
            if evaluation == True:
    
                val_loss, val_results = self.evaluate(val_dataloader)

                self.history['loss']['val_loss'].append(val_loss)
                for metric in val_results:
                    self.history['metrics'][metric].append(val_results[metric].cpu())
    
                time_elapsed = time.time() - t_start
    
                
                if val_results[val_metric] > self.best_metric:
                    self.best_metric = val_results[val_metric]
                    epochs_without_improvements = 0
                    
                    if save_best:
                        self.save_checkpoint('best_ckpt.pt', self.epoch)
                        
                self.best_metric = max(self.best_metric, val_results[val_metric])
                
                print(f"{epoch + 1:^7} | {'-':^7} | train_loss {avg_train_loss:^12.6f} | val_loss {val_loss:^10.6f} | precision {val_results['precision']:^9.4f} | recall {val_results['recall']:^9.4f} | specificity {val_results['specificity']:^9.4f} | f1 {val_results['f1']:^9.4f} | {time_elapsed:^9.2f}")
    
                if early_stopping:
                    if epochs_without_improvements == early_stopping:
                        break
                    
        self.save_checkpoint('last_ckpt.pt', self.epoch)


    def evaluate(self, val_dataloader):
    
        val_loss = []
        
        if self.task == 'binary':
            metrics = {
                'precision': torchmetrics.Precision(task='binary', threshold=0.5),
                'recall': torchmetrics.Recall(task='binary', threshold=0.5),
                'specificity': torchmetrics.Specificity(task='binary', threshold=0.5),
                'f1': torchmetrics.F1Score(task='binary', threshold=0.5)
            }
        else: 
            metrics = {
                'precision': torchmetrics.Precision(task='multiclass', num_classes=num_classes, average='macro'),
                'recall': torchmetrics.Recall(task='multiclass', num_classes=num_classes, average='macro'),
                'specificity': torchmetrics.Specificity(task='multiclass', num_classes=num_classes, average='macro'),
                'f1': torchmetrics.F1Score(task='multiclass', num_classes=num_classes, average='macro')
            }
        
        for metric in metrics.values():
            metric.to(self.device)
        
        self.eval()
        with torch.no_grad():
            with tqdm(total = len(val_dataloader)) as tq:
                for batch in val_dataloader:
                    
                    b_input_ids, b_attn_mask, b_scores, b_labels = tuple(t.to(self.device) for t in batch)
                    b_labels = b_labels.reshape(-1, 1)
                    
                    pred = self.forward(b_input_ids, b_scores, b_attn_mask)
            
                    loss = self.loss_fn(pred, b_labels)
                    val_loss.append(loss.item())
            
                    for metric in metrics.values():
                        metric.update(pred, b_labels)
                    
                    tq.update(1)
        
        results = {name: metric.compute() for name, metric in metrics.items()}
        
        for metric in metrics.values():
            metric.reset()
            
        return np.mean(val_loss), results
    
    def save_checkpoint(self, filename, epoch):
        state = {
            'epoch': epoch,
            'model': self,
            }
        torch.save(state, filename)

    def set_loss_fn(self, loss_fn):
        self.loss_fn = loss_fn

    def set_optimizer(self, optimizer):
        self.optimizer = optimizer

    def set_scheduler(self, scheduler):
        self.scheduler = scheduler

    
    def load_best_model(self, path='bert_binary/best_ckpt.pt'):
        ckpt = torch.load(path)
        print(f'Model loaded from epoch {ckpt["epoch"]}')
        return ckpt['model']

    def load_last_model(self, path='bert_binary/last_ckpt.pt'):
        ckpt = torch.load(path)
        print(f'Model loaded from epoch {ckpt["epoch"]}')
        return ckpt['model']

    def plot_history(self):
        fig, ax = plt.subplots(figsize=(12,4))
        ax.plot(self.history['loss']['train_loss'],label='train_loss')
        ax.plot(self.history['loss']['val_loss'],label='val_loss')
        ax.set_xlabel('epoch')
        ax.set_ylabel('loss')
        ax.legend()
        plt.show()

        metric_keys = list(self.history['metrics'].keys())
        num_metrics = len(metric_keys)
        
        fig, axs = plt.subplots(len(metric_keys), 1, figsize=(12,4*num_metrics))
        axs = axs.ravel()
        for ax, metric_name in zip(axs, metric_keys):
            ax.plot(self.history['metrics'][metric_name])
            ax.set_xlabel('epoch')
            ax.set_ylabel(metric_name)
            
        plt.show()
        
        

In [16]:
model_binary = BERTClassifier(bert)

In [ ]:
model_binary = model_binary.load_best_model('bert_binary/best_ckpt.pt')

Model loaded from epoch 40


In [18]:
pred1 = model_binary.predict(dataloader1, thresh=0.5)
pred2 = model_binary.predict(dataloader2, thresh=0.5)

100%|██████████| 50/50 [00:05<00:00,  9.72it/s]


In [19]:
df1['problem1_stated'] = pred1.astype(bool)
df2['problem2_stated'] = pred2.astype(bool)

In [ ]:
class BERTMultiClassifier(nn.Module):
    
    def __init__(self, bert, task='binary', n_classes=None, device='cuda'):

        super(BERTMultiClassifier, self).__init__()
        
        self.bert = bert
        self.task = task
        self.n_classes = n_classes

        self.classifier = nn.Sequential(
            nn.Linear(768, 256),
            nn.SiLU(),
            nn.Dropout(0.1),
            
            nn.Linear(256, 256),
            nn.SiLU(),
            nn.Dropout(0.1),
            

            nn.Linear(256, 512),
            nn.SiLU(),
            nn.Dropout(0.05),

            nn.Linear(512, 512),
            nn.SiLU(),
            nn.Dropout(0.05),
            
            nn.Linear(512, n_classes)
        )
        
        self.best_metric = 0
        self.epoch = 0
        self.history = {'loss' : {'train_loss' : [], 'val_loss' : []}, 
                        'metrics' : {'precision': [], 'recall': [], 'specificity' : [], 'f1' : []}}
        self.device = device
        self.to(device)

    def forward(self, x, s, mask):
  
        _, x = self.bert(x, attention_mask=mask).values()
        x = self.classifier(x)

        return x

    def predict(self, data, thresh=0.5):
        
        self.eval()
        predictions = []
        
        with tqdm(total = len(data)) as tq:
            for batch in data:
                b_data = tuple(t.to(self.device) for t in batch)
                if len(b_data) == 4:
                    b_input_ids, b_attn_mask, b_scores, b_labels = b_data
                else:
                    b_input_ids, b_attn_mask, b_scores = b_data
                with torch.no_grad():
                    preds = self.forward(b_input_ids, b_scores, b_attn_mask).cpu()
                predictions.extend(preds)
                tq.update(1)
                
        predictions = np.array(predictions)

        if self.task == 'binary':
            return (predictions.reshape(-1) > thresh).astype(int)
            
        return predictions.argmax(axis=1).astype(int)

    def fit(self, train_dataloader, val_dataloader=None, val_metric = 'f1', epochs=5, early_stopping = None, evaluation=True, save_best=True):

        epochs_without_improvements = 0
        
        for epoch in range(self.epoch, self.epoch + epochs):
    
            t_start = time.time()
    
            total_loss = 0.
    
            self.train()
    
            with tqdm(total = len(train_dataloader)) as tq:
                
                for step, batch in enumerate(train_dataloader):
                    
                    self.optimizer.zero_grad()
                    b_input_ids, b_attn_mask, b_scores, b_labels = tuple(t.to(self.device) for t in batch)
                    # b_labels = b_labels.reshape(-1,1)
                    pred = self.forward(b_input_ids, b_scores, b_attn_mask)
                    loss = self.loss_fn(pred, b_labels)
                    loss_value = loss.item()
                    total_loss += loss_value
        
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(self.parameters(), 1.0)
                    self.optimizer.step()
                    self.scheduler.step()
                    tq.update(1)
                    tq.set_description("Loss %s" % loss_value)
            
    
            avg_train_loss = total_loss / len(train_dataloader)
            
            self.history['loss']['train_loss'].append(avg_train_loss)

            self.epoch += 1

            epochs_without_improvements += 1
    
            if evaluation == True:
    
                val_loss, val_results = self.evaluate(val_dataloader)

                self.history['loss']['val_loss'].append(val_loss)
                for metric in val_results:
                    self.history['metrics'][metric].append(val_results[metric].cpu())
    
                time_elapsed = time.time() - t_start
    
                
                if val_results[val_metric] > self.best_metric:
                    self.best_metric = val_results[val_metric]
                    epochs_without_improvements = 0
                    
                    if save_best:
                        self.save_checkpoint('best_ckpt.pt', self.epoch)
                        
                self.best_metric = max(self.best_metric, val_results[val_metric])
                
                print(f"{epoch + 1:^7} | {'-':^7} | train_loss {avg_train_loss:^12.6f} | val_loss {val_loss:^10.6f} | precision {val_results['precision']:^9.4f} | recall {val_results['recall']:^9.4f} | specificity {val_results['specificity']:^9.4f} | f1 {val_results['f1']:^9.4f} | {time_elapsed:^9.2f}")
    
                if early_stopping:
                    if epochs_without_improvements == early_stopping:
                        break
                    
        self.save_checkpoint('last_ckpt.pt', self.epoch)


    def evaluate(self, val_dataloader):
    
        val_loss = []
        
        if self.task == 'binary':
            metrics = {
                'precision': torchmetrics.Precision(task='binary', threshold=0.5),
                'recall': torchmetrics.Recall(task='binary', threshold=0.5),
                'specificity': torchmetrics.Specificity(task='binary', threshold=0.5),
                'f1': torchmetrics.F1Score(task='binary', threshold=0.5)
            }
        else: 
            metrics = {
                'precision': torchmetrics.Precision(task='multiclass', num_classes=self.n_classes, average='macro'),
                'recall': torchmetrics.Recall(task='multiclass', num_classes=self.n_classes, average='macro'),
                'specificity': torchmetrics.Specificity(task='multiclass', num_classes=self.n_classes, average='macro'),
                'f1': torchmetrics.F1Score(task='multiclass', num_classes=self.n_classes, average='macro')
            }
        
        for metric in metrics.values():
            metric.to(self.device)
        
        self.eval()
        with torch.no_grad():
            with tqdm(total = len(val_dataloader)) as tq:
                for batch in val_dataloader:
                    
                    b_input_ids, b_attn_mask, b_scores, b_labels = tuple(t.to(self.device) for t in batch)
                    # b_labels = b_labels.reshape(-1, 1)
                    
                    pred = self.forward(b_input_ids, b_scores, b_attn_mask)
            
                    loss = self.loss_fn(pred, b_labels)
                    val_loss.append(loss.item())
            
                    for metric in metrics.values():
                        metric.update(pred, b_labels)
                    
                    tq.update(1)
        
        results = {name: metric.compute() for name, metric in metrics.items()}
        
        for metric in metrics.values():
            metric.reset()
            
        return np.mean(val_loss), results
    
    def save_checkpoint(self, filename, epoch):
        state = {
            'epoch': epoch,
            'model': self,
            }
        torch.save(state, filename)

    def set_loss_fn(self, loss_fn):
        self.loss_fn = loss_fn

    def set_optimizer(self, optimizer):
        self.optimizer = optimizer

    def set_scheduler(self, scheduler):
        self.scheduler = scheduler

    
    def load_best_model(self, path='bert_multiclass/best_ckpt.pt'):
        ckpt = torch.load(path)
        print(f'Model loaded from epoch {ckpt["epoch"]}')
        return ckpt['model']

    def load_last_model(self, path='bert_multiclass/last_ckpt.pt'):
        ckpt = torch.load(path)
        print(f'Model loaded from epoch {ckpt["epoch"]}')
        return ckpt['model']

    def plot_history(self):
        fig, ax = plt.subplots(figsize=(12,4))
        ax.plot(self.history['loss']['train_loss'],label='train_loss')
        ax.plot(self.history['loss']['val_loss'],label='val_loss')
        ax.set_xlabel('epoch')
        ax.set_ylabel('loss')
        ax.legend()
        plt.show()

        metric_keys = list(self.history['metrics'].keys())
        num_metrics = len(metric_keys)
        
        fig, axs = plt.subplots(len(metric_keys), 1, figsize=(12,4*num_metrics))
        axs = axs.ravel()
        for ax, metric_name in zip(axs, metric_keys):
            ax.plot(self.history['metrics'][metric_name])
            ax.set_xlabel('epoch')
            ax.set_ylabel(metric_name)
            
        plt.show()

In [ ]:
model = BERTMultiClassifier(None, task='multiclass', n_classes=28)

model = model.load_best_model('bert_multiclass/best_ckpt.pt')

Model loaded from epoch 238


In [22]:
df1p = df1[df1.problem1_stated == True]
df2p = df2[df2.problem2_stated == True]

In [25]:
inputs1, masks1 = preprocessing_for_bert(np.array(df1p['A1']))
inputs2, masks2 = preprocessing_for_bert(np.array(df2p['A2']))

scores1 = torch.tensor(np.array(df1p['Score']), dtype=torch.float32)
scores2 = torch.tensor(np.array(df2p['Score']), dtype=torch.float32)

batch_size = 64

data1 = TensorDataset(inputs1, masks1, scores1)
sampler1 = SequentialSampler(data1)
dataloader1 = DataLoader(data1, sampler=sampler1, batch_size=batch_size)

data2 = TensorDataset(inputs2, masks2, scores2)
sampler2 = SequentialSampler(data2)
dataloader2 = DataLoader(data2, sampler=sampler2, batch_size=batch_size)

100%|██████████| 2092/2092 [00:01<00:00, 1496.60it/s]


In [26]:
pred1 = model.predict(dataloader1)
pred2 = model.predict(dataloader2)

100%|██████████| 33/33 [00:03<00:00,  9.74it/s]


In [27]:
df1p['class1'] = pred1 + 2
df2p['class2'] = pred2 + 2

In [28]:
df1p['cat1'] = df1p['class1'].apply(get_category_name)
df2p['cat2'] = df2p['class2'].apply(get_category_name)

In [30]:
df1 = df1.join(df1p['cat1'], how='left')

In [31]:
df2 = df2.join(df2p['cat2'], how='left')

In [32]:
df1['cat1'] = df1['cat1'].fillna('ок / нет ответа')

In [33]:
df2['cat2'] = df2['cat2'].fillna('ок / нет ответа')

In [34]:
df = df.join(df1['cat1'], how='left')
df = df.join(df2['cat2'], how='left')

In [35]:
df['cat1'] = df['cat1'].fillna('нет ответа')
df['cat2'] = df['cat2'].fillna('нет ответа')

In [36]:
df.loc[:, ['A1', 'cat1', 'A2', 'cat2']].to_csv('df_processed.csv', index=False)